In [1]:
import os
import numpy as np
import nibabel as nib
import tensorflow as tf
from tensorflow.keras import backend as K
from tqdm import tqdm
 
# =========================================================================
# Paths
# =========================================================================
MODEL_PATH = "E:/Radiomics/best_unet_model.keras"
DATA_DIR = "E:/Radiomics/data"
MASK_OUT_DIR = "E:/Radiomics/corrected_masks"
MODALITIES = ["t1", "t1ce", "t2", "flair"]
 
PATCH_SIZE = (112, 112, 112)
INFERENCE_STRIDE = (56, 56, 56)   # must match what you evaluated with during training
NUM_CLASSES = 4
 
os.makedirs(MASK_OUT_DIR, exist_ok=True)

In [2]:
# =========================================================================
# FIX #3: loss/metric definitions — copied verbatim from the TRAINING
# script so custom_objects here matches exactly what the model was
# trained and evaluated with.
# =========================================================================
SMOOTH = 1e-6
 
 
def dice_coef(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    return (2. * inter + SMOOTH) / (K.sum(y_true_f) + K.sum(y_pred_f) + SMOOTH)
 
 
def combined_dice_ce_loss(y_true, y_pred):
    y_true = tf.squeeze(y_true, axis=-1)
    num_classes = K.int_shape(y_pred)[-1]
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    dice_sum = 0.0
    for c in range(1, num_classes):
        dice_sum += dice_coef(y_true_oh[..., c], y_pred[..., c])
    dice_mean = dice_sum / tf.cast(num_classes - 1, tf.float32)
    ce = K.categorical_crossentropy(y_true_oh, y_pred)
    return 0.5 * (1.0 - dice_mean) + 0.5 * K.mean(ce)
 
 
def iou_coef(y_true, y_pred, smooth=1e-6):
    y_true = tf.squeeze(y_true, axis=-1)
    num_classes = K.int_shape(y_pred)[-1]
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    y_pred_arg = K.argmax(y_pred, axis=-1)
    y_pred_oh = tf.one_hot(y_pred_arg, depth=num_classes)
    intersection = K.sum(y_true_oh * y_pred_oh, axis=[0, 1, 2, 3])
    union = K.sum(y_true_oh + y_pred_oh, axis=[0, 1, 2, 3]) - intersection
    return K.mean((intersection + smooth) / (union + smooth))
 
 
def dice_whole_tumor(y_true, y_pred):
    y_true = tf.squeeze(y_true, axis=-1)
    num_classes = K.int_shape(y_pred)[-1]
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    y_pred_arg = K.argmax(y_pred, axis=-1)
    y_true_wt = K.cast(K.any(y_true_oh[..., 1:], axis=-1), 'float32')
    y_pred_wt = K.cast(K.any(tf.stack([tf.equal(y_pred_arg, 1), tf.equal(y_pred_arg, 2),
                                        tf.equal(y_pred_arg, 3)], axis=-1), axis=-1), 'float32')
    return dice_coef(y_true_wt, y_pred_wt)
 
 
def dice_tumor_core(y_true, y_pred):
    y_true = tf.squeeze(y_true, axis=-1)
    num_classes = K.int_shape(y_pred)[-1]
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    y_pred_arg = K.argmax(y_pred, axis=-1)
    y_true_tc = K.cast(K.any(tf.stack([y_true_oh[..., 1], y_true_oh[..., 3]], axis=-1), axis=-1), 'float32')
    y_pred_tc = K.cast(K.any(tf.stack([tf.equal(y_pred_arg, 1), tf.equal(y_pred_arg, 3)], axis=-1), axis=-1), 'float32')
    return dice_coef(y_true_tc, y_pred_tc)
 
 
def dice_enhancing_tumor(y_true, y_pred):
    y_true = tf.squeeze(y_true, axis=-1)
    num_classes = K.int_shape(y_pred)[-1]
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    y_pred_arg = K.argmax(y_pred, axis=-1)
    y_true_et = y_true_oh[..., 3]
    y_pred_et = K.cast(tf.equal(y_pred_arg, 3), 'float32')
    return dice_coef(y_true_et, y_pred_et)
 
 
custom_objects = {
    'combined_dice_ce_loss': combined_dice_ce_loss,
    'dice_whole_tumor': dice_whole_tumor,
    'dice_tumor_core': dice_tumor_core,
    'dice_enhancing_tumor': dice_enhancing_tumor,
    'iou_coef': iou_coef,
}
 
 
# =========================================================================
# FIX #1: identical preprocessing to training — image-based brain-bbox
# crop (never uses labels), plus the same sliding-window inference used
# for training-side evaluation. No resize/squash anywhere.
# =========================================================================
def normalize_nonzero(vol: np.ndarray) -> np.ndarray:
    vol = vol.astype(np.float32)
    nz = vol[vol != 0]
    if nz.size > 0:
        vol = (vol - nz.mean()) / (nz.std() + 1e-8)
    return vol
 
 
def crop_to_brain_bbox_with_coords(image_4ch: np.ndarray, margin: int = 5,
                                    intensity_percentile: float = 1.0):
    """
    Same logic as training's crop_to_brain_bbox(), but also returns the
    bounding box coordinates so the prediction can be placed back into
    the original full-volume shape afterward.
    """
    signal = np.abs(image_4ch).sum(axis=-1)
    thresh = np.percentile(signal[signal > 0], intensity_percentile) if np.any(signal > 0) else 0
    fg = signal > thresh
 
    coords = np.array(np.nonzero(fg))
    if coords.size == 0:
        zmin, ymin, xmin = 0, 0, 0
        zmax, ymax, xmax = np.array(image_4ch.shape[:3]) - 1
    else:
        zmin, ymin, xmin = coords.min(axis=1)
        zmax, ymax, xmax = coords.max(axis=1)
        zmin = max(zmin - margin, 0)
        ymin = max(ymin - margin, 0)
        xmin = max(xmin - margin, 0)
        zmax = min(zmax + margin, image_4ch.shape[0] - 1)
        ymax = min(ymax + margin, image_4ch.shape[1] - 1)
        xmax = min(xmax + margin, image_4ch.shape[2] - 1)
 
    img_c = image_4ch[zmin:zmax + 1, ymin:ymax + 1, xmin:xmax + 1]
    bbox = (int(zmin), int(zmax), int(ymin), int(ymax), int(xmin), int(xmax))
    return img_c, bbox
 
 
def pad_to_min_shape(arr, min_shape, pad_value=0):
    pad_width = []
    for i, s in enumerate(arr.shape[:3]):
        needed = max(0, min_shape[i] - s)
        before = needed // 2
        after = needed - before
        pad_width.append((before, after))
    if arr.ndim == 4:
        pad_width.append((0, 0))
    return np.pad(arr, pad_width, mode='constant', constant_values=pad_value)
 
 
def sliding_window_predict(model, volume_4ch, patch_size, stride, num_classes):
    """Same sliding-window logic as the training/evaluation pipeline."""
    orig_shape = volume_4ch.shape[:3]
    ph, pw, pd = patch_size
    sh, sw, sd = stride
 
    padded = pad_to_min_shape(volume_4ch, patch_size, pad_value=0)
    H, W, D = padded.shape[:3]
 
    def stops(dim_size, patch, step):
        if dim_size <= patch:
            return [0]
        s = list(range(0, dim_size - patch + 1, step))
        if s[-1] != dim_size - patch:
            s.append(dim_size - patch)
        return s
 
    y_stops = stops(H, ph, sh)
    x_stops = stops(W, pw, sw)
    z_stops = stops(D, pd, sd)
 
    prob_sum = np.zeros((H, W, D, num_classes), dtype=np.float32)
    count = np.zeros((H, W, D, 1), dtype=np.float32)
 
    for y in y_stops:
        for x in x_stops:
            for z in z_stops:
                patch = padded[y:y + ph, x:x + pw, z:z + pd]
                pred = model.predict(patch[np.newaxis, ...].astype(np.float32), verbose=0)[0]
                prob_sum[y:y + ph, x:x + pw, z:z + pd] += pred
                count[y:y + ph, x:x + pw, z:z + pd] += 1.0
 
    count[count == 0] = 1.0
    avg_prob = prob_sum / count
    label_full = np.argmax(avg_prob, axis=-1).astype(np.uint8)
    return label_full[:orig_shape[0], :orig_shape[1], :orig_shape[2]]
 
 
def region_mask_np(label_volume: np.ndarray, region: str) -> np.ndarray:
    if region == "WT":
        return np.isin(label_volume, [1, 2, 3])
    if region == "TC":
        return np.isin(label_volume, [1, 3])
    if region == "ET":
        return label_volume == 3
    raise ValueError(f"Unknown region: {region}")

In [3]:
# =========================================================================
# Main inference loop
# =========================================================================
print("Loading model...")
model = tf.keras.models.load_model(MODEL_PATH, custom_objects=custom_objects, compile=False)
print("Model loaded successfully")
print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)

# FIX #2: `patients` was undefined before — build it from DATA_DIR.
patients = sorted([
    d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))
])
print(f"Found {len(patients)} patients in {DATA_DIR}")

for pid in tqdm(patients):
    # --- RESUME LOGIC START ---
    out_path = os.path.join(MASK_OUT_DIR, f"{pid}_pred_seg.nii.gz")
    if os.path.exists(out_path):
        # Optional: If you still want to print volumes for skipped patients, 
        # you would need to load the existing mask here. Otherwise, just skip:
        continue 
    # --- RESUME LOGIC END ---

    pdir = os.path.join(DATA_DIR, pid)
    vols = []
    affine = None
    orig_shape = None

    for mod in MODALITIES:
        file_path = os.path.join(pdir, f"{pid}_{mod}.nii.gz")
        nii = nib.load(file_path)
        vol = nii.get_fdata(dtype=np.float32)
        if orig_shape is None:
            orig_shape = vol.shape
            affine = nii.affine
        vols.append(normalize_nonzero(vol))

    image_4ch = np.stack(vols, axis=-1)

    # FIX #1: crop to brain bbox (identical to training), not resize.
    image_cropped, bbox = crop_to_brain_bbox_with_coords(image_4ch)
    zmin, zmax, ymin, ymax, xmin, xmax = bbox

    # FIX #1: sliding-window inference over the cropped volume,
    # identical to training-side evaluation — no single-shot resize.
    seg_cropped = sliding_window_predict(model, image_cropped, PATCH_SIZE, INFERENCE_STRIDE, NUM_CLASSES)

    # Place the cropped-region prediction back into the ORIGINAL
    # full-volume shape. Everything outside the crop is background
    # (label 0), since crop_to_brain_bbox only trims near-zero
    # background in the first place.
    seg_full = np.zeros(orig_shape, dtype=np.uint8)
    seg_full[zmin:zmax + 1, ymin:ymax + 1, xmin:xmax + 1] = seg_cropped

    nib.save(nib.Nifti1Image(seg_full, affine), out_path)

    # FIX #4: correct region volumes — WT/TC/ET are COMBINATIONS of
    # classes, not individual classes 1/2/3.
    voxel_volume = abs(np.linalg.det(affine[:3, :3]))
    for region in ("WT", "TC", "ET"):
        voxels = region_mask_np(seg_full, region).sum()
        volume_mm3 = voxels * voxel_volume
        print(f"  {pid} — {region}: {volume_mm3:.2f} mm^3")


Loading model...
Model loaded successfully
Input shape : (None, 112, 112, 112, 4)
Output shape: (None, 112, 112, 112, 4)
Found 1250 patients in E:/Radiomics/data


 41%|████      | 509/1250 [05:15<09:29,  1.30it/s]  

  BraTS2021_00728 — WT: 31411.00 mm^3
  BraTS2021_00728 — TC: 758.00 mm^3
  BraTS2021_00728 — ET: 754.00 mm^3


 41%|████      | 510/1250 [11:32<25:30,  2.07s/it]

  BraTS2021_00729 — WT: 13925.00 mm^3
  BraTS2021_00729 — TC: 6595.00 mm^3
  BraTS2021_00729 — ET: 5167.00 mm^3


 41%|████      | 511/1250 [16:36<43:46,  3.55s/it]

... (223 outputs left out)

 50%|████▉     | 623/1250 [27:23:38<2744:46:52, 15759.51s/it]

  BraTS2021_01038 — WT: 85500.00 mm^3
  BraTS2021_01038 — TC: 42792.00 mm^3
  BraTS2021_01038 — ET: 33999.00 mm^3


 50%|████▉     | 623/1250 [27:24:24<27:34:57, 158.37s/it]    


KeyboardInterrupt: 